In [5]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split

INPUT_ROOT = Path("/kaggle/input")

OUTPUT_DIR = Path(
    "/kaggle/working/data/splits"
)

RANDOM_STATE = 42

print("Libraries loaded successfully.")
print("Kaggle input:", INPUT_ROOT)


Libraries loaded successfully.
Kaggle input: /kaggle/input


In [6]:

# List datasets attached to the notebook

for item in INPUT_ROOT.iterdir():
    print(item)

# Find satellite images
image_candidates = sorted(
    INPUT_ROOT.rglob("*_sat.jpg")
)

print("\nNumber of satellite images found:")
print(len(image_candidates))

print("\nFirst 5 image paths:")

for path in image_candidates[:5]:
    print(path)


/kaggle/input/datasets

Number of satellite images found:
8570

First 5 image paths:
/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/test/100393_sat.jpg
/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/test/100995_sat.jpg
/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/test/101982_sat.jpg
/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/test/10206_sat.jpg
/kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/test/102240_sat.jpg


In [8]:

pairs = []
missing_masks = []

for image_path in image_candidates:

    # Find its matching mask
    mask_path = image_path.with_name(
        image_path.name.replace(
            "_sat.jpg",
            "_mask.png"
        )
    )

    if not mask_path.exists():
        missing_masks.append(image_path.name)
        continue

    # Check image and mask dimensions
    with Image.open(image_path) as image:
        image_size = image.size

    with Image.open(mask_path) as mask:
        mask_size = mask.size

    if image_size != mask_size:
        raise ValueError(
            f"Dimension mismatch: {image_path.name}"
        )

    pairs.append({
        "sample_id": image_path.name.replace(
            "_sat.jpg", ""
        ),
        "image_filename": image_path.name,
        "mask_filename": mask_path.name,
        "image_path": str(image_path),
        "mask_path": str(mask_path)
    })

if not pairs:
    raise RuntimeError("No valid image-mask pairs found.")

pairs_df = pd.DataFrame(pairs)

print("Valid image-mask pairs:", len(pairs_df))
print("Images without masks:", len(missing_masks))

print("\nFirst 5 matched pairs:")
display(
    pairs_df[
        ["sample_id", "image_filename", "mask_filename"]
    ].head()
)


Valid image-mask pairs: 6226
Images without masks: 2344

First 5 matched pairs:


,sample_id,image_filename,mask_filename
0,100034,100034_sat.jpg,100034_mask.png
1,100081,100081_sat.jpg,100081_mask.png
2,100129,100129_sat.jpg,100129_mask.png
3,100703,100703_sat.jpg,100703_mask.png
4,100712,100712_sat.jpg,100712_mask.png


In [9]:

# Split 70% for training and 30% remaining

train_df, remaining_df = train_test_split(
    pairs_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Split remaining data equally:
# 15% validation and 15% test

validation_df, test_df = train_test_split(
    remaining_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    shuffle=True
)

# Reset row indexes
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Training:", len(train_df))
print("Validation:", len(validation_df))
print("Test:", len(test_df))
print("Total:", len(pairs_df))


Training: 4358
Validation: 934
Test: 934
Total: 6226


In [10]:

train_ids = set(train_df["sample_id"])
validation_ids = set(validation_df["sample_id"])
test_ids = set(test_df["sample_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

assert (
    len(train_df) +
    len(validation_df) +
    len(test_df)
) == len(pairs_df)

print("No overlap found between splits.")
print("All image-mask pairs are accounted for.")


No overlap found between splits.
All image-mask pairs are accounted for.


In [11]:

def preprocess_pair(
    image_path,
    mask_path,
    target_size=None
):
    """
    Preprocess one satellite image and its road mask.

    Image:
        RGB format
        Pixel values normalized to [0, 1]

    Mask:
        Grayscale
        Binary labels: road=1, background=0

    target_size:
        Optional tuple (width, height).
    """

    # Load satellite image in RGB format
    with Image.open(image_path) as image:
        image = ImageOps.exif_transpose(
            image
        ).convert("RGB")

    # Load road mask in grayscale
    with Image.open(mask_path) as mask:
        mask = ImageOps.exif_transpose(
            mask
        ).convert("L")

    # Verify dimensions
    if image.size != mask.size:
        raise ValueError(
            "Image and mask sizes do not match."
        )

    # Resize 
    if target_size is not None:

        image = image.resize(
            target_size,
            Image.Resampling.BILINEAR
        )

        mask = mask.resize(
            target_size,
            Image.Resampling.NEAREST
        )

    # Convert image to a NumPy array
    image_array = np.asarray(
        image,
        dtype=np.float32
    ) / 255.0

    # Convert mask to binary labels
    mask_array = np.asarray(
        mask,
        dtype=np.uint8
    )

    binary_mask = (
        mask_array >= 128
    ).astype(np.float32)

    # Add one channel to the mask
    binary_mask = binary_mask[..., np.newaxis]

    return image_array, binary_mask


In [12]:

sample = pairs_df.iloc[0]

sample_image, sample_mask = preprocess_pair(
    sample["image_path"],
    sample["mask_path"]
)

print("Image shape:", sample_image.shape)
print("Mask shape:", sample_mask.shape)

print("Image minimum:", sample_image.min())
print("Image maximum:", sample_image.max())

print("Mask unique values:", np.unique(sample_mask))


Image shape: (1024, 1024, 3)
Mask shape: (1024, 1024, 1)
Image minimum: 0.011764706
Image maximum: 0.9764706
Mask unique values: [0. 1.]


In [13]:

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

splits = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

manifest_frames = []

for split_name, frame in splits.items():

    # Save satellite image filenames only
    list_path = OUTPUT_DIR / f"{split_name}.txt"

    frame["image_filename"].to_csv(
        list_path,
        index=False,
        header=False
    )

    # Prepare combined manifest
    part = frame[
        [
            "sample_id",
            "image_filename",
            "mask_filename"
        ]
    ].copy()

    part["split"] = split_name

    manifest_frames.append(part)

# Save combined CSV
manifest = pd.concat(
    manifest_frames,
    ignore_index=True
)

manifest = manifest[
    [
        "split",
        "sample_id",
        "image_filename",
        "mask_filename"
    ]
]

manifest.to_csv(
    OUTPUT_DIR / "manifest.csv",
    index=False
)

# Save summary
summary = {
    "random_state": RANDOM_STATE,
    "ratios": {
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15
    },
    "total_pairs": len(pairs_df),
    "counts": {
        name: len(frame)
        for name, frame in splits.items()
    },
    "image_suffix": "_sat.jpg",
    "mask_suffix": "_mask.png",
    "mask_threshold": 128
}

with open(
    OUTPUT_DIR / "split_summary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(summary, file, indent=4)

print("All split files saved!")
print("Output folder:", OUTPUT_DIR)

for file in sorted(OUTPUT_DIR.iterdir()):
    print(file.name)


All split files saved!
Output folder: /kaggle/working/data/splits
manifest.csv
split_summary.json
test.txt
train.txt
validation.txt


In [14]:
import shutil

zip_path = shutil.make_archive(
    "/kaggle/working/member2_splits",
    "zip",
    root_dir=OUTPUT_DIR
)

print("ZIP created:", zip_path)

ZIP created: /kaggle/working/member2_splits.zip
